In [ ]:
#| hide
from fastcore.nbio import read_nb as _read_raw_nb
from fastcore.nbio import write_nb as _write_raw_nb

from nbskill.convert import convert
from nbskill.edit import edit_notebook
from nbskill.execute import exec_nb
from nbskill.foundation import demo_path, remove_demo_path, tool_notebook, write_demo_file
from nbskill.mcp import create_mcp
from nbskill.read import context
from nbskill.review import diff_nb

In [ ]:
#| hide
demo_nb = demo_path("index_tool.ipynb")
_write_raw_nb(tool_notebook(), demo_nb)
nb = _read_raw_nb(demo_nb)
answer_cell = next(cell for cell in nb.cells if "answer =" in cell.source)

# nbskill

`nbskill` is notebook-aware tooling for agents and maintainers working in nbdev projects. It keeps notebooks as the source of truth while giving automation stable, reviewable operations for reading, editing, executing, converting, and reviewing them.

The package is useful when plain JSON edits are too blunt and generated Python edits would bypass the notebook narrative.

## Why it exists

A notebook is code, prose, metadata, outputs, and execution state in one file. That is powerful for nbdev, but awkward for coding agents: a small change can hit the wrong cell, keep stale outputs, or skip the documentation cell that explains the exported function below it.

`nbskill` provides a narrow toolchain around the notebook shape itself. The normal loop is: inspect a focused context, apply a structured notebook edit, then run the smallest execution or review check that proves the change.

## What it offers

| Need | Tooling |
| --- | --- |
| Find the right context | `context` reads a project, notebook, chapter, cell id, or symbol without raw JSON noise. |
| Make focused edits | `edit_notebook`, `write_nb`, and `update_cell` preserve cell structure and clear stale outputs. |
| Verify behavior | `exec_nb` runs the notebook in project context, with a check-only mode for review loops. |
| Review changes | `diff_nb`, `style_check`, and `doctor` focus on code cells, notebook hygiene, and tool diagnostics. |
| Move code into nbdev | `convert` turns Python files or packages into nbdev notebooks. |
| Serve agents | `nbskill_mcp` exposes the same workflow as MCP tools. |

## Install and connect

From a checkout, install the project and start the MCP server with the package scripts:

```bash
uv sync
uv run nbskill_mcp
```

The same functions are available as Python APIs and CLI commands such as `context`, `write_nb`, `exec_nb`, `diff_nb`, `style_check`, `convert`, and `agent_workbench`.

## A notebook-aware tour

The examples below create one disposable demo notebook and run real `nbskill` operations against it. Cells that would otherwise modify it use `dry_run=True`.

In [ ]:
demo_nb.name

### Read focused context

`context` gives an agent the smallest useful view of a project, notebook, chapter, cell, or symbol. Symbol context is especially useful in nbdev repos because it links the implementation back to the notebook that owns it.

In [ ]:
read_result = context(answer_cell.id, scope=str(demo_nb))

Cell context: /Users/macbook/Projects/nbskill/tests/fixtures/nbskill_tool_fixture.ipynb Cell id=fixture-answer: code

Cell
Cell id=fixture-answer idx=1 [code] (example)
answer = 42


### Plan edits with stable cell ids

`edit_notebook` applies deterministic operations against stable cell ids. Here the operation is a dry run: it computes the change and affected cell ids without writing to the fixture.

In [ ]:
line_edit = dict(op="replace_lines", cell_id=answer_cell.id, start_line=1, end_line=1, replacement_lines=["answer = x * 10"])
edit_result = edit_notebook(demo_nb, [line_edit], dry_run=True, auto_feedback=False)

edit_result["dry_run"], edit_result["changed"], edit_result["affected_cell_ids"]

(True, True, ['fixture-answer'])

In [ ]:
print("\n".join(edit_result["text"].splitlines()[:8]))

edit_notebook dry_run: 1 changed operation(s), 1 affected cell(s)

--- fixture-answer:before
+++ fixture-answer:after
@@ -1 +1 @@
-answer = 42
+answer = x * 10


### Refactor across cells

Notebook-wide text replacements are useful for small, mechanical refactors. With `dry_run=True`, the same operation becomes a safe preview for review.

In [ ]:
rename_edit = dict(op="replace_text", target="all", old="answer", new="result")
rename_result = edit_notebook(demo_nb, [rename_edit], dry_run=True, auto_feedback=False)

rename_result["dry_run"], len(rename_result["affected_cell_ids"])

(True, 5)

In [ ]:
rename_result["text"].splitlines()[0]

'edit_notebook dry_run: 5 changed operation(s), 5 affected cell(s)'

### Execute as a notebook

`exec_nb` runs cells in notebook order with local project imports available. In review loops, `check_only=True` executes in memory without writing outputs back to disk.

In [ ]:
_ = exec_nb(str(demo_nb), timeout=10, allow_new=True, check_only=True, show_output=True)

Executed /Users/macbook/Projects/nbskill/tests/fixtures/nbskill_tool_fixture.ipynb -> not written (safe) (check_only=True) (timeout=10s)
--- output id=fixture-example ---
84


### Review code-cell changes

`diff_nb` ignores notebook metadata churn and focuses on code-cell source. Passing `ref_a=None` and `ref_b=None` reviews a notebook as current content, which is handy for examples and generated fixtures.

In [ ]:
_ = diff_nb(str(demo_nb), ref_a=None, ref_b=None)

No code cell changes


In [ ]:
#| hide
remove_demo_path(demo_nb)

### Convert Python into an nbdev notebook

`convert` can bootstrap a notebook from existing Python. That gives an agent or maintainer a starting point for moving code into nbdev's notebook-first workflow.

In [ ]:
sample_source = "def greet(name):\n    return f'Hello {name}'\n"
with write_demo_file("index_sample.py", sample_source) as sample_py:
    converted_nb = demo_path("index_sample_converted.ipynb")
    _ = convert(str(sample_py), dest=str(converted_nb), dry_run=True)
    print(converted_nb.name)

### Serve the workflow through MCP

The MCP server wraps the same notebook-aware operations as tools. The server layer stays thin: it captures output, serializes notebook writes, and delegates behavior back to the notebook-defined functions.

## CLI and MCP workflow

A typical agent session stays small and reversible:

```bash
context nbs/02_write.ipynb
context write_nb --scope nbs/02_write.ipynb
# apply a focused MCP edit_notebook operation, preferably replace_lines or a small insert_cells
# write Markdown rationale, smallest code, visible example, hidden focused test
diff_nb nbs/02_write.ipynb
exec_nb nbs/02_write.ipynb --check_only
style_check nbs/02_write.ipynb --changed_only
```

Keep imports in their own cell. Do not combine them with definitions, examples, or test setup, because documentation builds can run import cells in an isolated namespace.

For a behavior change, add or revise a focused notebook test before the implementation when a useful reproducer exists. Run the focused check before the edit, then run it again after. Assert the promised behavior, not formatting, `repr`, or incidental order.

Use `nbskill_mcp` when the client can call MCP tools directly. Prefer MCP edits for source notebooks and avoid changing generated Python unless you are deliberately debugging export output. Use `style_check(changed_only=True)` after edits so the feedback highlights problems introduced by the current diff.

After exporting a library change, verify it in a clean process or restart the active kernel. Reloading one module can leave direct imports and patched classes stale.

A good notebook change should read like a small story:

```python
# Markdown cell before the code:
# "We normalize empty titles here because imported notebooks may omit them."

def normalized_title(title):
    return title.strip() or "Untitled"

normalized_title("  Demo  ")

assert normalized_title("  ") == "Untitled"
```

## How this repo is organized

The notebooks in `nbs/` follow the toolchain itself:

1. `00_foundation.ipynb` defines shared notebook, path, git, and nbdev helpers.
2. `01_read.ipynb` makes notebooks readable by project, chapter, cell, and symbol.
3. `02_write.ipynb` and `02_edit.ipynb` provide safe cell writes and structured edits.
4. `03_execute.ipynb` executes notebooks in the local project context.
5. `04_review.ipynb` provides notebook diffs, style checks, and diagnostics.
6. `05_convert.ipynb` converts Python sources into nbdev notebooks.
7. `06_skill.ipynb` builds the bundled agent skill.
8. `07_mcp.ipynb` exposes the workflow as MCP tools.
9. `08_edit_interactive.ipynb` and `11_agent_workbench.ipynb` run bounded edit loops.
10. `09_parallel.ipynb` keeps concurrent notebook operations orderly.
11. `10_graph.ipynb` and `12_knowledge.ipynb` build symbol and reference context.
12. `13_cli.ipynb` keeps command-line wrappers close to the public APIs.

## Development workflow

Work notebook-first: read the relevant context, edit the source notebook, run a focused execution or diff check, and let nbdev regenerate Python from the notebook. That is the standard this project is built to support.